# 03 · Join Sofascore + Capology — England Premier League 25/26 (snapshot 20260428)

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2025/26 de la Premier League inglesa**.

⚠️ **Nota sobre el snapshot:** la temporada 25/26 está aún en curso. Se trabaja con
una foto fija de Sofascore (`df_england_2526_snapshot_20260428.csv`). Este notebook
deberá reejecutarse con los datos definitivos cuando finalice la liga, generando
entonces el master sin sufijo de fecha (`master_england_2526.csv`).

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_england_2526_snapshot_20260428.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_england_2526.csv').copy()

print(f'Sofascore (snapshot):  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:              {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore (snapshot):  525 jugadores | 117 columnas
Capology:              706 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   brighton hove albion
   leeds united
   newcastle united
   tottenham hotspur
   west ham united

En Capology pero no en Sofascore:
   brighton
   leeds
   newcastle
   tottenham
   west ham


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brighton':'brighton hove albion',
            'leeds':'leeds united',
            'newcastle':'newcastle united',
            'tottenham':'tottenham hotspur',
            'west ham':'west ham united'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')

✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 471/525 (89.7%)
Sin emparejar: 54


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          7
Revisión media    (0.75 ≤ score < 0.90):   10
Revisión estricta (0.50 ≤ score < 0.75):   25
Revisión muy est. (score < 0.50):           12


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
33,Romain Esse,Crystal Palace,romain esse,1.000
24,Jota Silva,Nottingham Forest,jota silva,1.000
20,David Möller Wolfe,Wolverhampton,david mller wolfe,0.971
29,Christian Norgaard,Arsenal,christian nrgaard,0.971
2,Martin Odegaard,Arsenal,martin degaard,0.966
7,Yéremy Pino,Crystal Palace,yeremi pino,0.909
32,Joshua King,Fulham,josh king,0.900


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
14,Yehor Yarmoliuk,Brentford,yegor yarmolyuk,0.867
11,Andy Robertson,Liverpool,andrew robertson,0.867
5,Đorđe Petrović,Bournemouth,djordje petrovic,0.857
9,Idrissa Gana Gueye,Everton,idrissa gueye,0.839
16,Savinho,Manchester City,savio,0.833
19,Jamie Gittens,Chelsea,jamie bynoe gittens,0.812
17,Florentino Luís,Burnley,florentino,0.800
37,Divine Mukasa,Manchester City,divin mubama,0.800
1,Stefan Ortega,Nottingham Forest,stefan ortega moreno,0.788
8,Pape Matar Sarr,Tottenham Hotspur,pape sarr,0.750


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['divine mukasa'

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 9 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
12,Hannibal Mejbri,Burnley,hannibal,0.696
0,Tom Edozie,Wolverhampton,tommy doyle,0.667
35,Hamed Junior Traorè,Bournemouth,eli junior kroupi,0.611
50,Tyler Fletcher,Manchester United,tyler fredricson,0.600
22,Guido Rodríguez,West Ham United,gideon kodua,0.593
30,George Hemmings,Aston Villa,tyrone mings,0.593
41,Pablo Felipe,West Ham United,pablo,0.588
31,Hwang Hee-chan,Wolverhampton,hee chan hwang,0.571
46,Igor Júlio,Brighton & Hove Albion,igor,0.571
47,Callum Marshall,West Ham United,callum wilson,0.571


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['hannibal mejbri',
                    'pablo felipe',
                    'hwang hee chan',
                    'igor julio',
                    'jair',
                    'john victor',
                    'andre'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')

Aceptados del nivel bajo: 7


### 7.4 Revisión muy estricta (score < 0.50)

Candidatos con muy baja similitud. Por defecto ninguno se acepta.
Añadir a `ACCEPT_VERY_LOW_FUZZY` los que se confirmen manualmente.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
4,Lucas Paquetá,West Ham United,lukasz fabianski,0.483
18,Mohamadou Kanté,West Ham United,mads hermansen,0.483
42,Bradley Burrowes,Aston Villa,morgan rogers,0.483
40,Harry Howell,Brighton & Hove Albion,danny welbeck,0.480
25,Jamaldeen Jimoh,Aston Villa,samuel iling junior,0.471
44,Jamaal Lascelles,Newcastle United,aaron ramsdale,0.467
39,Joel Drakes-Thomas,Crystal Palace,jesurun rak sakyi,0.457
53,Joe Knight,Brighton & Hove Albion,joel veltman,0.455
49,Nayef Aguerd,West Ham United,ezra mayers,0.435
13,Jun'ai Byfield,Tottenham Hotspur,lucas bergvall,0.429


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')

Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 492/525 (93.7%)
Sin salario:     33


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 33


,player,team,minutesPlayed,appearances,goals,assists
0,George Hemmings,Aston Villa,25,2,0,0
1,Bradley Burrowes,Aston Villa,16,1,0,0
2,Jamaldeen Jimoh,Aston Villa,8,1,0,0
3,Hamed Junior Traorè,Bournemouth,16,1,0,1
4,Harry Howell,Brighton & Hove Albion,93,3,0,0
5,Nehemiah Oriola,Brighton & Hove Albion,1,1,0,0
6,Joe Knight,Brighton & Hove Albion,1,1,0,0
7,Oliver Sonne,Burnley,165,7,1,0
8,Romain Esse,Crystal Palace,54,4,0,0
9,Joel Drakes-Thomas,Crystal Palace,25,2,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Aston Villa  —  SF sin salario:


,player,minutesPlayed
0,Bradley Burrowes,16
1,George Hemmings,25
2,Jamaldeen Jimoh,8


  CG plantilla completa:


,player,player_norm
0,Alysson,alysson
1,Amadou Onana,amadou onana
2,Andrés García,andres garcia
3,Ben Broggio,ben broggio
4,Boubacar Kamara,boubacar kamara
5,Donyell Malen,donyell malen
6,Douglas Luiz,douglas luiz
7,Emiliano Buendía,emiliano buendia
8,Emiliano Martínez,emiliano martinez
9,Evann Guessand,evann guessand



  Bournemouth  —  SF sin salario:


...

,player,player_norm
0,Aaron Wan-Bissaka,aaron wan bissaka
1,Adama Traoré,adama traore
2,Alphonse Areola,alphonse areola
3,Axel Disasi,axel disasi
4,Callum Wilson,callum wilson
5,Crysencio Summerville,crysencio summerville
6,Edson Álvarez,edson alvarez
7,El Hadji Malick Diouf,el hadji malick diouf
8,Ezra Mayers,ezra mayers
9,Freddie Potts,freddie potts



  Wolverhampton  —  SF sin salario:


,player,minutesPlayed
0,Tom Edozie,25


  CG plantilla completa:


,player,player_norm
0,Adam Armstrong,adam armstrong
1,André Trindade,andre trindade
2,Angel Gomes,angel gomes
3,Boubacar Traoré,boubacar traore
4,Daniel Bentley,daniel bentley
5,David Møller Wolfe,david mller wolfe
6,Dexter Lembikisa,dexter lembikisa
7,Emmanuel Agbadou,emmanuel agbadou
8,Enso González,enso gonzalez
9,Fer López,fer lopez


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')

Matches manuales definidos: 0


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 492/525 (93.7%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

⚠️ Nombre con sufijo `_snapshot_20260428` para diferenciar del master definitivo
que se generará al cierre de la temporada (`master_england_2526.csv`).

In [21]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_england_2526_snapshot_20260428.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_england_2526_snapshot_20260428.csv
   Jugadores totales:  525
   Con salario:        492
   Sin salario (NaN):  33
   Columnas:           122
